<a href="https://colab.research.google.com/github/hpsj2712/atelie-generativo-individual-homerio/blob/main/notebooks/01_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conexão e configuração do repositório

In [ ]:
import os
if not os.path.exists('atelie-generativo-individual-homerio'):
  !git clone https://github.com/hpsj2712/atelie-generativo-individual-homerio.git
else:
  print('Repository already cloned.')

Cloning into 'atelie-generativo-individual-homerio'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 112 (delta 22), reused 16 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 128.10 MiB | 14.84 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [ ]:
cd atelie-generativo-individual-homerio

/content/atelie-generativo-individual-homerio


Inicio da codigificação de testes:

In [ ]:
!pip -q install transformers pillow

In [ ]:
import csv

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image, ExifTags
from datetime import datetime


arquivo_legendas = "dados/legendas.txt"
arquivo_fontes = "dados/fontes.csv"
autor = "Homerio Junior"
licenca = "autoral"
repositorio = "repositorio particular"

In [ ]:
proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained(
"Salesforce/blip-image-captioning-base").to("cuda")


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [ ]:
# Cabeçalhos do CSV
csv_headers = ["nome da imagem", "url", "autor", "licença", "data coleta"]

# Abre o arquivo CSV para escrita
with open(arquivo_fontes, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()

    # Seu loop original (adaptado para incluir o CSV)
    with open(arquivo_legendas, "w") as f:
        for i in range(1, 41):
            img_name = f"img_{i:03d}.jpg"
            img_filename = f"dados/{img_name}"
            img_raw = Image.open(img_filename)

            try:
                # Verficair Exif e se há data
                data_coleta = None
                try:
                    exif_data = img_raw._getexif()

                    if exif_data:
                        # Inverte o dicionário para buscar pelo nome da tag
                        # ExifTags.TAGS mapeia ID -> Nome (ex: 36867 -> 'DateTimeOriginal')
                        for tag_id, value in exif_data.items():
                            tag_name = ExifTags.TAGS.get(tag_id)
                            if tag_name == 'DateTimeOriginal':
                                # Formato EXIF: 'YYYY:MM:DD HH:MM:SS'
                                data_coleta = datetime.strptime(value, '%Y:%m:%d %H:%M:%S').strftime('%d/%m/%Y %H:%M:%S')
                                break
                except Exception as e:
                    print(f"Aviso: Não foi possível ler EXIF de {img_name} (pode não existir): {e}")

                # Se o EXIF falhar ou não existir, você pode manter um fallback para a data do arquivo:
                if not data_coleta:
                    timestamp = os.path.getmtime(img_filename)
                    data_coleta = datetime.fromtimestamp(timestamp).strftime('%d/%m/%Y %H:%M:%S')

                # 2. Processamento de legenda Imagem (BLIP)
                img = img_raw.convert("RGB")
                inputs = proc(img, return_tensors="pt").to("cuda")
                out = blip.generate(**inputs, max_new_tokens=40)
                legenda = proc.decode(out[0], skip_special_tokens=True)

                # 3. Escrever no TXT
                f.write(f"{img_name}: estilo_arquitetura_modernista_brasilia, {legenda}\n")

                # 4. Escrever no CSV
                writer.writerow({
                    "nome da imagem": img_name,
                    "url": repositorio,
                    "autor": autor,
                    "licença": licenca,
                    "data coleta": data_coleta
                })

                print(f"Processado: {img_name}")

            except FileNotFoundError:
                print(f"Atenção: Imagem {img_name} não encontrada.")
            except Exception as e:
                print(f"Erro ao processar {img_name}: {e}")

print(f"Processo concluído. Arquivos salvos em 'dados/'.")

Processado: img_001.jpg
Processado: img_002.jpg
Processado: img_003.jpg
Processado: img_004.jpg
Processado: img_005.jpg
Processado: img_006.jpg
Processado: img_007.jpg
Processado: img_008.jpg
Processado: img_009.jpg
Processado: img_010.jpg
Processado: img_011.jpg
Processado: img_012.jpg
Processado: img_013.jpg
Processado: img_014.jpg
Processado: img_015.jpg
Processado: img_016.jpg
Processado: img_017.jpg
Processado: img_018.jpg
Processado: img_019.jpg
Processado: img_020.jpg
Processado: img_021.jpg
Processado: img_022.jpg
Processado: img_023.jpg
Processado: img_024.jpg
Processado: img_025.jpg
Processado: img_026.jpg
Processado: img_027.jpg
Processado: img_028.jpg
Processado: img_029.jpg
Processado: img_030.jpg
Processado: img_031.jpg
Processado: img_032.jpg
Processado: img_033.jpg
Processado: img_034.jpg
Processado: img_035.jpg
Processado: img_036.jpg
Processado: img_037.jpg
Processado: img_038.jpg
Processado: img_039.jpg
Processado: img_040.jpg
Processo concluído. Arquivos salvos em '

In [ ]:
from datasets import Dataset, Image
import pandas as pd
import os
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('hugging-face')

login(token)

repo_path = "/content/temp_repo"
!git clone https://github.com/hpsj2712/atelie-generativo-individual-homerio {repo_path}

# 3. Carregue as legendas
legendas_dict = {}
legendas_file = os.path.join(repo_path, 'dados', 'legendas.txt')

with open(legendas_file, 'r', encoding='utf-8') as f:
    for line in f:
        if ":" in line:
            parts = line.split(":", 1)
            file_name = parts[0].strip()
            caption = parts[1].strip()

            image_path = os.path.join(repo_path, 'dados', file_name)

            if os.path.exists(image_path):
                legendas_dict[image_path] = caption


df = pd.DataFrame(list(legendas_dict.items()), columns=['image', 'text'])
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("image", Image())


dataset.push_to_hub("homerio/atelie_estilo_brasilia")


!rm -rf {repo_path}